# Capítulo 8: Casos de Uso en la Industria

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/caracena/apunte-analitica-textual/blob/main/capitulos/clase8-casos-de-uso.ipynb)

## Objetivos de aprendizaje

- Conocer aplicaciones reales de analítica textual en diversas industrias.
- Implementar prototipos funcionales para legaltech, salud, finanzas y atención al cliente.
- Integrar las técnicas aprendidas en los capítulos anteriores en soluciones completas.

## 8.1 Panorama de aplicaciones

La analítica textual tiene aplicaciones en prácticamente todas las industrias que generan o procesan documentos.

| Industria | Problema | Técnica | Impacto |
|-----------|----------|---------|--------|
| **Legal** | Revisión de contratos | Clasificación + NER | Reducción de 80% en tiempo de revisión |
| **Salud** | Extracción de datos clínicos | NER + Clasificación | Mejor calidad de registros |
| **Finanzas** | Análisis de sentimiento de mercado | Sentimiento + Clustering | Señales de trading |
| **Atención al cliente** | Enrutamiento de tickets | Clasificación | Menor tiempo de respuesta |

In [ ]:
import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

## 8.2 Caso 1: LegalTech — Clasificación de cláusulas contractuales

En el sector legal, la revisión de contratos es una tarea intensiva. Un sistema de clasificación puede identificar automáticamente el tipo de cada cláusula.

In [ ]:
# Dataset simulado de cláusulas contractuales
clausulas = [
    # Confidencialidad
    "Las partes se comprometen a mantener en estricta reserva toda información confidencial",
    "No se podrá divulgar información sensible a terceros sin autorización previa por escrito",
    "La obligación de confidencialidad se mantendrá vigente por un plazo de 5 años",
    "Toda información técnica comercial y financiera será tratada como confidencial",
    # Terminación
    "Cualquiera de las partes podrá poner término al contrato con 30 días de preaviso",
    "El contrato se resolverá de pleno derecho en caso de incumplimiento grave",
    "La terminación anticipada dará lugar a una indemnización equivalente a tres meses",
    "El contrato podrá ser rescindido por mutuo acuerdo de las partes en cualquier momento",
    # Pago
    "El precio del servicio será de 500 UF mensuales pagaderas dentro de los primeros 5 días",
    "El pago se realizará mediante transferencia bancaria a la cuenta del prestador",
    "En caso de mora se aplicará un interés del 1.5% mensual sobre el saldo adeudado",
    "Las facturas deberán ser emitidas y enviadas antes del último día hábil de cada mes",
    # Jurisdicción
    "Para todos los efectos legales las partes fijan domicilio en Santiago de Chile",
    "Las controversias serán resueltas por arbitraje según las reglas del CAM Santiago",
    "Las partes se someten a la jurisdicción de los tribunales ordinarios de justicia",
    "Cualquier disputa será resuelta mediante mediación antes de recurrir a la vía judicial",
]

tipos = (["confidencialidad"] * 4 + ["terminación"] * 4 +
         ["pago"] * 4 + ["jurisdicción"] * 4)

# Entrenar clasificador
X_train, X_test, y_train, y_test = train_test_split(
    clausulas, tipos, test_size=0.25, random_state=42, stratify=tipos
)

clasificador_legal = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LinearSVC(random_state=42, max_iter=10000))
])

clasificador_legal.fit(X_train, y_train)
y_pred = clasificador_legal.predict(X_test)

print("=== Clasificación de Cláusulas Contractuales ===")
print(classification_report(y_test, y_pred))

# Probar con nuevas cláusulas
nuevas = [
    "El monto total será pagado en 12 cuotas iguales y sucesivas",
    "Ambas partes acuerdan no revelar los términos de este acuerdo",
    "El contrato tendrá una duración de un año y podrá ser renovado tácitamente"
]

print("Predicciones:")
for clausula, pred in zip(nuevas, clasificador_legal.predict(nuevas)):
    print(f"  [{pred.upper()}] {clausula}")

## 8.3 Caso 2: Salud — Extracción de información de reportes clínicos

En salud, la extracción automática de entidades (medicamentos, diagnósticos, dosis) de textos clínicos permite estructurar información que de otro modo permanecería atrapada en texto libre.

In [ ]:
# Extracción de información clínica con regex
reportes_clinicos = [
    "Paciente de 45 años, sexo masculino. Diagnóstico: hipertensión arterial. "
    "Se indica Losartán 50mg cada 12 horas. Próximo control en 30 días.",
    
    "Mujer de 62 años con diabetes mellitus tipo 2. Se prescribe Metformina 850mg "
    "con cada comida. Glicemia en ayunas: 180 mg/dL. Control en 15 días.",
    
    "Paciente de 33 años consulta por cefalea crónica. Antecedente de migraña. "
    "Se indica Sumatriptán 50mg al inicio del episodio. Derivación a neurología.",
]

def extraer_info_clinica(texto):
    """Extrae información estructurada de un reporte clínico."""
    info = {}
    
    # Edad
    edad = re.search(r'(\d+)\s*años', texto)
    info['edad'] = int(edad.group(1)) if edad else None
    
    # Sexo
    if re.search(r'masculino|hombre|varón', texto, re.IGNORECASE):
        info['sexo'] = 'M'
    elif re.search(r'femenino|mujer', texto, re.IGNORECASE):
        info['sexo'] = 'F'
    else:
        info['sexo'] = None
    
    # Diagnóstico
    dx = re.search(r'[Dd]iagnóstico:?\s*([^.]+)', texto)
    if not dx:
        dx = re.search(r'con\s+([\w\s]+?(?:tipo\s+\d)?)\.', texto)
    info['diagnostico'] = dx.group(1).strip() if dx else None
    
    # Medicamentos y dosis
    medicamentos = re.findall(r'([A-Z][a-záéíóú]+)\s+(\d+\s*mg)', texto)
    info['medicamentos'] = [{'nombre': m[0], 'dosis': m[1]} for m in medicamentos]
    
    # Próximo control
    control = re.search(r'[Cc]ontrol\s+en\s+(\d+)\s*días', texto)
    info['proximo_control_dias'] = int(control.group(1)) if control else None
    
    return info

print("=== Extracción de Información Clínica ===\n")
for i, reporte in enumerate(reportes_clinicos, 1):
    info = extraer_info_clinica(reporte)
    print(f"Reporte {i}:")
    print(f"  Edad: {info['edad']} años")
    print(f"  Sexo: {info['sexo']}")
    print(f"  Diagnóstico: {info['diagnostico']}")
    for med in info['medicamentos']:
        print(f"  Medicamento: {med['nombre']} {med['dosis']}")
    print(f"  Próximo control: {info['proximo_control_dias']} días\n")

## 8.4 Caso 3: Finanzas — Análisis de sentimiento de noticias

En finanzas, el sentimiento de las noticias puede ser una señal predictiva del comportamiento del mercado.

In [ ]:
# Análisis de sentimiento financiero
noticias_financieras = [
    # Positivas
    "Las acciones del sector tecnológico alcanzan máximos históricos impulsadas por resultados trimestrales",
    "El banco central mantiene la tasa de interés señalando estabilidad económica",
    "La empresa reporta un crecimiento del 25% en ingresos superando expectativas del mercado",
    "Inversores muestran confianza con flujos récord hacia fondos de renta variable",
    "El PIB creció un 4.2% en el último trimestre impulsado por exportaciones",
    "La compañía anuncia un plan de expansión internacional y contratación masiva",
    # Negativas
    "Crisis en el sector inmobiliario genera preocupación entre los inversores",
    "Las acciones caen un 15% tras el anuncio de investigación por fraude",
    "La inflación alcanza su nivel más alto en una década presionando al consumo",
    "La empresa anuncia despidos masivos y cierre de operaciones en tres países",
    "Fuga de capitales y devaluación de la moneda generan incertidumbre económica",
    "El mercado registra su peor semana desde la pandemia por temores de recesión",
]

sentimientos = ["positivo"] * 6 + ["negativo"] * 6

# Entrenar modelo
modelo_financiero = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2))),
    ('clf', MultinomialNB())
])

modelo_financiero.fit(noticias_financieras, sentimientos)

# Analizar nuevas noticias
nuevas_noticias = [
    "La bolsa sube un 3% tras acuerdo comercial entre las principales economías",
    "Quiebra de banco regional genera pánico en los mercados financieros",
    "Startup chilena recibe inversión millonaria para expandir sus operaciones",
    "Desplome del precio del cobre amenaza la economía exportadora"
]

print("=== Análisis de Sentimiento Financiero ===\n")
predicciones = modelo_financiero.predict(nuevas_noticias)

for noticia, sent in zip(nuevas_noticias, predicciones):
    indicador = "▲" if sent == "positivo" else "▼"
    print(f"  {indicador} [{sent.upper()}] {noticia}")

In [ ]:
# Simulación de señal de trading basada en sentimiento
np.random.seed(42)
dias = 20
sentimiento_diario = np.random.choice(["positivo", "negativo"], size=dias, p=[0.6, 0.4])
scores = np.where(sentimiento_diario == "positivo", 1, -1).astype(float)
scores += np.random.normal(0, 0.3, dias)  # ruido

# Señal acumulada
señal_acumulada = np.cumsum(scores)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

colores = ['#2ecc71' if s == 'positivo' else '#e74c3c' for s in sentimiento_diario]
ax1.bar(range(dias), scores, color=colores, alpha=0.7)
ax1.set_ylabel('Score de sentimiento')
ax1.set_title('Sentimiento diario de noticias financieras')
ax1.axhline(y=0, color='black', linewidth=0.5)
ax1.grid(True, alpha=0.3)

ax2.plot(range(dias), señal_acumulada, 'b-o', linewidth=2, markersize=4)
ax2.fill_between(range(dias), señal_acumulada, alpha=0.1)
ax2.set_xlabel('Día')
ax2.set_ylabel('Señal acumulada')
ax2.set_title('Señal de trading acumulada')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8.5 Caso 4: Atención al cliente — Clasificación y enrutamiento de tickets

Un sistema de clasificación automática puede categorizar tickets de soporte y enrutarlos al equipo correcto, reduciendo tiempos de respuesta.

In [ ]:
# Clasificación de tickets de soporte
tickets = [
    # Facturación
    "Me cobraron dos veces el mismo servicio en mi tarjeta de crédito",
    "Necesito una nota de crédito por la factura del mes pasado",
    "¿Cuándo se emite la factura del servicio contratado?",
    "El monto cobrado no coincide con el plan que tengo contratado",
    "Solicito la devolución del cobro indebido realizado en mi cuenta",
    # Soporte técnico
    "La plataforma no carga y muestra un error 500 en el navegador",
    "No puedo iniciar sesión aunque ingreso la contraseña correcta",
    "El reporte se genera incompleto le faltan las últimas columnas",
    "La aplicación móvil se cierra inesperadamente al abrir el dashboard",
    "La integración con la API dejó de funcionar después de la actualización",
    # Ventas
    "Quisiera información sobre los planes empresariales y sus precios",
    "¿Tienen descuento para organizaciones sin fines de lucro?",
    "Necesito una demo del producto para presentar a mi equipo directivo",
    "¿Es posible contratar el servicio solo por 6 meses como prueba?",
    "Estamos evaluando migrar desde otro proveedor cuál es el proceso",
]

categorias = ["facturación"] * 5 + ["soporte_técnico"] * 5 + ["ventas"] * 5

# Entrenar modelo
modelo_tickets = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2))),
    ('clf', LinearSVC(random_state=42, max_iter=10000))
])

modelo_tickets.fit(tickets, categorias)

# Clasificar nuevos tickets
nuevos_tickets = [
    "Quiero cancelar mi suscripción y que me devuelvan lo que pagué",
    "El sistema está muy lento desde esta mañana no puedo trabajar",
    "¿Cuántos usuarios permite el plan profesional?",
    "Me llegó una factura con datos incorrectos de razón social"
]

equipos = {
    "facturación": "Equipo Finanzas",
    "soporte_técnico": "Equipo Ingeniería",
    "ventas": "Equipo Comercial"
}

print("=== Enrutamiento de Tickets de Soporte ===\n")
for ticket, cat in zip(nuevos_tickets, modelo_tickets.predict(nuevos_tickets)):
    print(f"  Ticket: {ticket}")
    print(f"  → Categoría: {cat} → Asignado a: {equipos[cat]}\n")

## 8.6 Caso 5: Descubrimiento de temas en redes sociales

Combinando TF-IDF con clustering podemos descubrir automáticamente los temas de conversación en redes sociales.

In [ ]:
# Clustering de publicaciones en redes sociales
publicaciones = [
    "Increíble gol de último minuto en el clásico universitario",
    "La nueva actualización del iPhone trae funciones de IA generativa",
    "Receta fácil de pastel de chocolate para el fin de semana",
    "El equipo de fútbol logró la clasificación a la final continental",
    "Review del nuevo Samsung Galaxy con cámara de 200 megapíxeles",
    "Los mejores restaurantes de comida japonesa en Santiago",
    "Se confirma el fichaje del delantero estrella por 50 millones",
    "Tutorial para configurar tu smart home con asistentes de voz",
    "Top 5 recetas saludables para preparar en menos de 30 minutos",
    "Gran partido de tenis entre los dos mejores del ranking mundial",
    "Comparativa de los mejores notebooks para programadores en 2025",
    "Los secretos de la cocina italiana tradicional según chef premiado",
]

# Vectorizar y clusterizar
tfidf = TfidfVectorizer(max_features=50)
X = tfidf.fit_transform(publicaciones)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X)

# Palabras clave por cluster
terminos = tfidf.get_feature_names_out()

print("=== Temas Descubiertos en Redes Sociales ===\n")
for k in range(3):
    centroide = kmeans.cluster_centers_[k]
    top_terminos = [terminos[i] for i in centroide.argsort()[::-1][:4]]
    print(f"Tema {k+1} (palabras clave: {', '.join(top_terminos)}):")
    for i, pub in enumerate(publicaciones):
        if clusters[i] == k:
            print(f"  • {pub}")
    print()

## 8.7 Consideraciones prácticas

### Desafíos comunes en proyectos reales

1. **Datos insuficientes**: Técnicas como data augmentation, few-shot learning o transferencia de aprendizaje.
2. **Datos desbalanceados**: Oversampling, undersampling o ajuste de pesos en el modelo.
3. **Calidad de datos**: Ruido, errores ortográficos, formato inconsistente.
4. **Privacidad**: Datos sensibles (salud, finanzas) requieren anonimización.
5. **Multilingüismo**: Modelos multilingües o pipelines por idioma.
6. **Escalabilidad**: Procesamiento de millones de documentos requiere optimización.

### Buenas prácticas

- Comenzar con un baseline simple (TF-IDF + Naive Bayes) antes de modelos complejos.
- Evaluar con métricas relevantes para el negocio, no solo accuracy.
- Involucrar a expertos del dominio en la definición de categorías y validación.
- Monitorear el rendimiento en producción y re-entrenar periódicamente.
- Documentar el pipeline completo para reproducibilidad.

## Resumen del curso

A lo largo de este curso hemos recorrido el espectro completo de la analítica textual:

| Capítulo | Tema | Técnica clave |
|----------|------|---------------|
| 1 | Introducción y limpieza | Regex, tokenización |
| 2 | Representación numérica | TF-IDF, VSM, similitud coseno |
| 3 | Clustering | K-Means, descubrimiento de temas |
| 4 | Clasificación | Naive Bayes, SVM, MLP |
| 5 | Word Embeddings | Word2Vec, FastText |
| 6 | Transformers y LLMs | BERT, GPT, fine-tuning |
| 7 | Agentes de IA | ReAct, herramientas, RAG |
| 8 | Casos de uso | Legaltech, salud, finanzas, soporte |

Estas técnicas forman un toolkit versátil que permite abordar una amplia variedad de problemas reales con datos textuales.